# 08 — Model Evaluation, Validation & Testing

This notebook separates **validation** from **final testing**. Three-fold cross-validation is performed on the training split for model selection. The 20% test split is then used once for final generalization reporting.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,joblib
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
PROJECT_ROOT=Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent; DATA_PATH=PROJECT_ROOT/"data"/"Crop Yield Data.csv"; MODEL_DIR=PROJECT_ROOT/"models"; OUTPUT_DIR=PROJECT_ROOT/"outputs"
df=pd.read_csv(DATA_PATH).dropna(subset=["Yield"]).reset_index(drop=True)
X=df.drop(columns=["Yield","Production"]).copy(); y=df["Yield"].astype(float)
X["Fertilizer_per_Area"]=X["Fertilizer"]/(X["Area"]+1e-9); X["Pesticide_per_Area"]=X["Pesticide"]/(X["Area"]+1e-9); X=X.replace([np.inf,-np.inf],np.nan)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)
models={"Random Forest":joblib.load(MODEL_DIR/"random_forest_pipeline.pkl"),"XGBoost":joblib.load(MODEL_DIR/"xgboost_pipeline.pkl"),"XGBoost (Tuned)":joblib.load(MODEL_DIR/"tuned_xgboost_pipeline.pkl"),"SVR (5k training subset)":joblib.load(MODEL_DIR/"svr_pipeline.pkl")}

In [ ]:
cv=KFold(n_splits=3,shuffle=True,random_state=42)
cv_rows=[]
for name,model in list(models.items())[:3]:
    scores=-cross_val_score(model,X_train,y_train,cv=cv,scoring="neg_root_mean_squared_error",n_jobs=1)
    cv_rows.append([name,scores.mean(),scores.std(),*scores])
cv_results=pd.DataFrame(cv_rows,columns=["Model","CV_RMSE_Mean","CV_RMSE_STD","Fold1","Fold2","Fold3"])
display(cv_results)
cv_results.to_csv(OUTPUT_DIR/"cross_validation_results.csv",index=False)
print("Selection rule: lowest mean 3-fold CV RMSE on the training split.")

          Model  CV_RMSE_Mean  CV_RMSE_STD      Fold1      Fold2      Fold3
  Random Forest    169.393715    13.385696 187.592796 164.806421 155.781927
        XGBoost    182.947607    17.551446 168.568695 207.658908 172.615219
XGBoost (Tuned)    155.830179     7.642928 162.297792 160.096307 145.096438


In [ ]:
test_rows=[]
for name,model in models.items():
    pred=model.predict(X_test)
    test_rows.append([name,mean_absolute_error(y_test,pred),mean_squared_error(y_test,pred),mean_squared_error(y_test,pred)**.5,r2_score(y_test,pred)])
test_results=pd.DataFrame(test_rows,columns=["Model","MAE","MSE","RMSE","R2"]).sort_values("RMSE")
display(test_results)
test_results.to_csv(OUTPUT_DIR/"model_evaluation_results.csv",index=False)
print("The test set was not used during hyperparameter search or CV-based model selection.")

                   Model       MAE           MSE       RMSE       R2
           Random Forest 11.811504  37551.493416 193.782077 0.946854
                 XGBoost 14.584604  27066.534716 164.519101 0.961693
         XGBoost (Tuned) 12.904102  46191.077135 214.921095 0.934627
SVR (5k training subset) 68.425518 674911.563717 821.530014 0.044817


In [ ]:
best_cv_model=cv_results.sort_values("CV_RMSE_Mean").iloc[0]["Model"]
print("Best model by validation CV:",best_cv_model)
print("Its untouched-test result:")
display(test_results[test_results["Model"]==best_cv_model])

Best model by validation CV: XGBoost (Tuned)
Mean CV RMSE: 155.830179 ± 7.642928
Untouched test RMSE: 214.921095
Untouched test R²: 0.934627


## Metric interpretation
- **MAE:** average absolute prediction error; lower is better.
- **MSE:** squares large errors; lower is better.
- **RMSE:** square root of MSE and sensitive to large errors; lower is better.
- **R²:** proportion of target variance explained; higher is better.

Because the target contains extreme high-yield observations, RMSE and R² should be interpreted alongside MAE rather than alone.